# Module 5.7 — Mixing Sync & Async

### Python Mastery · Track 5: Concurrency & Asynchronous Programming

Real programs are rarely all asynchronous. You often need to call ordinary blocking code (a synchronous library, a CPU-bound calculation) from within async code, and sometimes to drive async code from a synchronous program. Doing this correctly is essential, because a single blocking call can freeze the entire event loop. This module covers the bridges between the two worlds and the rule that protects the loop.

**Running async code in a notebook.** As in the previous async modules, the executable cells use top-level `await` rather than `asyncio.run`, because a notebook already runs an event loop.

### Learning objectives

After completing this module you will be able to:

1. Explain why a blocking call harms the event loop.
2. Run blocking functions from async code with `asyncio.to_thread`.
3. Use `loop.run_in_executor` for finer control, including process pools.
4. Combine blocking and async work in one workflow.
5. Recognise when to keep code synchronous instead.

**Prerequisites:** Tracks 1 to 4, and Modules 5.1 to 5.6.

---

## Part 1 · Why Blocking Harms the Event Loop

The event loop runs everything on one thread, switching tasks only at `await` points. If a coroutine calls a slow **blocking** function (one that does not await, such as `time.sleep` or a heavy computation), the loop cannot switch away: every other task is frozen until the blocking call returns. The demonstration below contrasts a non-blocking await with a blocking call inside async code.

In [ ]:
import asyncio
import time

order = []

async def good_task(name):
    order.append(f"{name} start")
    await asyncio.sleep(0.05)         # non-blocking: yields to the loop
    order.append(f"{name} end")

# Two well-behaved tasks interleave because each yields at its await.
order.clear()
await asyncio.gather(good_task("A"), good_task("B"))
print("non-blocking, interleaved:", order)

In [ ]:
import asyncio
import time

order = []

async def bad_task(name):
    order.append(f"{name} start")
    time.sleep(0.05)                  # BLOCKING: does not yield, freezes the loop
    order.append(f"{name} end")

# Because time.sleep blocks, task A runs entirely before task B can begin.
order.clear()
await asyncio.gather(bad_task("A"), bad_task("B"))
print("blocking, no interleaving:", order)
print("note A fully finished before B started; the loop was frozen")

## Part 2 · `asyncio.to_thread` — The Simple Bridge

When you must call blocking code from async, the simplest fix is `asyncio.to_thread(func, *args)`. It runs the blocking function in a separate thread and gives you an awaitable, so the event loop stays free to run other tasks while the blocking work proceeds elsewhere. This is the recommended modern approach for offloading blocking input/output.

In [ ]:
import asyncio
import time

def blocking_io(name):
    """A synchronous function that blocks; imagine a library that is not async-aware."""
    time.sleep(0.05)
    return f"{name} done"

async def handle(name):
    # Run the blocking function in a worker thread; the loop is not frozen.
    result = await asyncio.to_thread(blocking_io, name)
    return result

import time as _t
start = _t.perf_counter()
results = await asyncio.gather(handle("A"), handle("B"), handle("C"))
elapsed = _t.perf_counter() - start

print("results:", results)
print(f"three 0.05s blocking calls took about {elapsed:.2f}s (overlapped via threads)")

## Part 3 · `loop.run_in_executor` — More Control

`asyncio.to_thread` is a convenient wrapper over a lower-level mechanism: `loop.run_in_executor(executor, func, *args)`. Passing `None` uses the default thread pool, equivalent to `to_thread`. Passing a `ProcessPoolExecutor` runs the work in a separate **process**, which is what you want for CPU-bound functions that the Global Interpreter Lock would otherwise serialise.

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

def blocking(label, n):
    total = 0
    for i in range(n):
        total += i
    return f"{label}: {total}"

loop = asyncio.get_running_loop()

# Using the default executor (None) is equivalent to asyncio.to_thread.
with ThreadPoolExecutor(max_workers=2) as pool:
    results = await asyncio.gather(
        loop.run_in_executor(pool, blocking, "X", 100_000),
        loop.run_in_executor(pool, blocking, "Y", 200_000),
    )
print(results)

> **Note on processes.** Offloading CPU-bound work to a `ProcessPoolExecutor` via `run_in_executor` is the correct pattern for true parallelism inside an async program. It requires the target function to be importable (picklable), as discussed in Module 5.2, so in production it is defined at module level in a `.py` file. The thread-pool form above is shown here because it runs cleanly in a notebook; the process form is identical except for the executor used.

## Part 4 · Combining Blocking and Async Work

A realistic workflow mixes both: some steps are naturally async (awaiting input/output), others call blocking libraries. The pattern is to keep the async parts as coroutines and wrap each blocking step with `to_thread`, so the whole pipeline runs without ever freezing the loop.

In [ ]:
import asyncio
import time

async def fetch(item):
    """An async step: awaiting input/output."""
    await asyncio.sleep(0.02)
    return f"raw-{item}"

def transform(data):
    """A blocking step: a synchronous, CPU-ish transformation."""
    time.sleep(0.02)
    return data.upper()

async def pipeline(item):
    raw = await fetch(item)                       # async step
    processed = await asyncio.to_thread(transform, raw)   # blocking step, offloaded
    return processed

results = await asyncio.gather(*(pipeline(i) for i in ["a", "b", "c"]))
print("pipeline results:", results)

## Part 5 · When to Stay Synchronous

Async is not always the right choice. It adds complexity and pays off mainly when a program juggles many concurrent input/output operations, such as a server handling thousands of connections. For a simple script, a command-line tool, or CPU-bound batch processing, plain synchronous code (with threads or processes where parallelism helps) is clearer and often just as fast. Choose async deliberately, for high-concurrency input/output, rather than by default.

In [ ]:
# A short decision guide, expressed as data for clarity.
guidance = [
    ("Many concurrent network calls / a server", "async (asyncio)"),
    ("A handful of blocking input/output calls", "threads or to_thread"),
    ("CPU-bound parallel computation", "multiprocessing / ProcessPoolExecutor"),
    ("A simple linear script", "plain synchronous code"),
]
for situation, choice in guidance:
    print(f"{situation:42} -> {choice}")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Offloading one blocking call (Easy)

In [ ]:
import asyncio
import time

def slow_square(n):
    time.sleep(0.02)
    return n * n

result = await asyncio.to_thread(slow_square, 9)
print("result:", result)

### Example 2 — Several blocking calls overlapped (Easy)

In [ ]:
import asyncio
import time

def work(n):
    time.sleep(0.02)
    return n + 100

results = await asyncio.gather(*(asyncio.to_thread(work, n) for n in range(4)))
print(results)

### Example 3 — Showing the loop stays responsive (Medium)

In [ ]:
import asyncio
import time

def blocking(n):
    time.sleep(0.05)
    return n

async def heartbeat(beats):
    ticks = []
    for i in range(beats):
        await asyncio.sleep(0.01)
        ticks.append(i)
    return ticks

# The heartbeat keeps ticking while the blocking work runs in a thread.
blocking_task = asyncio.create_task(asyncio.to_thread(blocking, 42))
heartbeat_task = asyncio.create_task(heartbeat(4))

value = await blocking_task
ticks = await heartbeat_task
print("blocking result:", value)
print("heartbeat ticked", len(ticks), "times during the blocking call")

### Example 4 — Using run_in_executor directly (Medium)

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

def compute(n):
    return sum(range(n))

loop = asyncio.get_running_loop()
with ThreadPoolExecutor() as pool:
    results = await asyncio.gather(
        loop.run_in_executor(pool, compute, 1000),
        loop.run_in_executor(pool, compute, 2000),
    )
print(results)

### Example 5 — A mixed pipeline with bounded concurrency (Difficult)

In [ ]:
import asyncio
import time

async def fetch(item):
    await asyncio.sleep(0.02)
    return item

def process(item):                       # blocking transformation
    time.sleep(0.02)
    return f"{item}-processed"

async def handle(item, semaphore):
    async with semaphore:                # cap how many run at once
        raw = await fetch(item)
        return await asyncio.to_thread(process, raw)

semaphore = asyncio.Semaphore(2)
results = await asyncio.gather(*(handle(i, semaphore) for i in ["a", "b", "c", "d"]))
print(results)

### Example 6 — Comparing blocking-in-loop versus offloaded (Difficult)

In [ ]:
import asyncio
import time

def blocking(n):
    time.sleep(0.03)
    return n

# Version 1: calling the blocking function directly serialises everything.
async def direct(n):
    blocking(n)                          # freezes the loop for each call
    return n

start = time.perf_counter()
await asyncio.gather(*(direct(n) for n in range(3)))
direct_time = time.perf_counter() - start

# Version 2: offloading lets the calls overlap.
async def offloaded(n):
    return await asyncio.to_thread(blocking, n)

start = time.perf_counter()
await asyncio.gather(*(offloaded(n) for n in range(3)))
offloaded_time = time.perf_counter() - start

print(f"blocking directly: about {direct_time:.2f}s (serialised)")
print(f"offloaded:         about {offloaded_time:.2f}s (overlapped)")

---

## Exercises

Use top-level `await` in your answers.

**Exercise 1 (Easy).** Use `asyncio.to_thread` to run a blocking function `slow_add(a, b)` (with a short `time.sleep`) and print the awaited result.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Use `asyncio.gather` with `asyncio.to_thread` to run a blocking `square(n)` for `n` in 1 to 4 concurrently, and print the results.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Demonstrate that the loop stays responsive: start a blocking call via `to_thread` as a task, and concurrently run a coroutine that appends three ticks (with short awaits). Print both the blocking result and the number of ticks.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Use `loop.run_in_executor` with a `ThreadPoolExecutor` to run two blocking summation functions concurrently and print their results.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Build a small pipeline where each item is first fetched (async, short sleep) and then processed by a blocking function (offloaded with `to_thread`), with concurrency capped at 2 via a semaphore. Run it over four items and print the results.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import asyncio, time

def slow_add(a, b):
    time.sleep(0.02)
    return a + b

print(await asyncio.to_thread(slow_add, 3, 4))

**Solution 2**

In [ ]:
import asyncio, time

def square(n):
    time.sleep(0.02)
    return n * n

print(await asyncio.gather(*(asyncio.to_thread(square, n) for n in range(1, 5))))

**Solution 3**

In [ ]:
import asyncio, time

def blocking():
    time.sleep(0.05)
    return "done"

async def ticker():
    ticks = []
    for i in range(3):
        await asyncio.sleep(0.01)
        ticks.append(i)
    return ticks

b = asyncio.create_task(asyncio.to_thread(blocking))
t = asyncio.create_task(ticker())
print("blocking:", await b)
print("ticks:", len(await t))

**Solution 4**

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

def total(n):
    return sum(range(n))

loop = asyncio.get_running_loop()
with ThreadPoolExecutor() as pool:
    print(await asyncio.gather(
        loop.run_in_executor(pool, total, 500),
        loop.run_in_executor(pool, total, 1500),
    ))

**Solution 5**

In [ ]:
import asyncio, time

async def fetch(item):
    await asyncio.sleep(0.02)
    return item

def process(item):
    time.sleep(0.02)
    return item.upper()

async def handle(item, sem):
    async with sem:
        raw = await fetch(item)
        return await asyncio.to_thread(process, raw)

sem = asyncio.Semaphore(2)
print(await asyncio.gather(*(handle(i, sem) for i in ["a", "b", "c", "d"])))

---

## Summary and Key Points

- The event loop runs on one thread and switches only at `await` points, so a blocking call (such as `time.sleep` or heavy computation) freezes every task until it returns.
- `asyncio.to_thread(func, *args)` runs a blocking function in a worker thread and returns an awaitable, keeping the loop free; it is the simplest way to offload blocking input/output.
- `loop.run_in_executor(executor, func, *args)` gives more control: `None` uses the default thread pool, while a `ProcessPoolExecutor` runs CPU-bound work in a separate process for true parallelism (the function must be importable).
- Mixed workflows keep async steps as coroutines and wrap each blocking step with `to_thread`, so the pipeline never freezes the loop.
- Async suits high-concurrency input/output; for simple scripts or CPU-bound batch work, plain synchronous code with threads or processes is clearer. Choose async deliberately.

### Track 5 complete

This concludes Track 5, Concurrency & Asynchronous Programming. You can now use threads for input/output concurrency, processes for CPU-bound parallelism, executors as a unified high-level interface, asyncio for large-scale cooperative concurrency, the advanced async tools for real workloads, the concepts behind distributed task systems, and the bridges between synchronous and asynchronous code. Track 6 builds on this with testing, tooling, and the practices that make code production-ready.